# 07_structured_generation_and_grammar: Constrained Structured Decoding

This notebook demonstrates regex-guided structured decoding using logit masking. We build a simple Finite State Machine (FSM) to validate next-token state changes and mask logits during decoding.

### Core Engineering Intuitions
- **Logit Masking Mechanics**: To force the LLM to output structured data (like JSON or SQL) matching a schema, structured serving engines (Outlines, XGrammar) track a Finite State Machine (FSM) built from the schema's regex. At each token selection step, the FSM lists the set of valid next character tokens. Invalid tokens in the vocabulary are masked by subtracting $-\infty$ from their raw logits. When Softmax is applied, the probability of selecting an invalid token becomes exactly $0\%$, guaranteeing schema compliance.
- **Parser Latency Overhead**: Building and compiling FSMs on-the-fly inside Python can add massive CPU overhead, bottlenecking throughput. Modern SOTA engines (XGrammar) bypass this by pre-compiling state transitions in C++ and caching masks, ensuring zero latency penalty on TPOT.

### Structured Generation Trade-offs (Pros & Cons)
- **Pros**: Guarantees that outputs strictly conform to JSON schemas or Pydantic formats; prevents API parser failures.
- **Cons**: Restricting candidate selection slightly limits generation creativity; FSM compiling can add TTFT latency if not cached.

In [1]:
import torch
import torch.nn.functional as F

class SimpleJSONFSM:
    def __init__(self, vocab):
        self.vocab = vocab
        self.state = 0

    def get_valid_tokens(self):
        if self.state == 0:
            return ["{"]
        elif self.state == 1:
            return ['"name"', '"age"']
        elif self.state == 2:
            return [":"]
        elif self.state == 3:
            return ["25", "30", '"John"']
        elif self.state == 4:
            return ["}"]
        return []

    def transition(self, token):
        if self.state == 0 and token == "{":
            self.state = 1
        elif self.state == 1 and token in ['"name"', '"age"']:
            self.state = 2
        elif self.state == 2 and token == ":":
            self.state = 3
        elif self.state == 3 and token in ["25", "30", '"John"']:
            self.state = 4
        elif self.state == 4 and token == "}":
            self.state = 5

In [2]:
vocab = ["{", '"name"', '"age"', ":", "25", "30", '"John"', "}", "abc", "xyz"]
fsm = SimpleJSONFSM(vocab)
torch.manual_seed(42)

generated_tokens = []
steps = 5

for step in range(steps):
    valid_toks = fsm.get_valid_tokens()
    mask = torch.ones(len(vocab)) * -float('inf')
    for idx, token in enumerate(vocab):
        if token in valid_toks:
            mask[idx] = 0.0
            
    logits = torch.randn(len(vocab)) * 2.0
    masked_logits = logits + mask
    
    probs = F.softmax(masked_logits, dim=-1)
    sel_idx = torch.multinomial(probs, num_samples=1).item()
    selected_token = vocab[sel_idx]
    
    generated_tokens.append(selected_token)
    fsm.transition(selected_token)

print("FSM Constrained Generated Output:")
print(" ".join(generated_tokens))

FSM Constrained Generated Output:
{ "name" : 30 }


### Output Explanation & Verification

- **Logit Masking**: Invalid tokens (like `'abc'` or `'xyz'`) were successfully masked out at each step by setting their logits to $-\infty$.
- **JSON Output Validation**: The generated token sequence: ` { "name" : 30 } ` conforms to the FSM grammar constraints. This proves that constrained logit masking guarantees structured JSON schema compliance.